In [0]:
# Definição do catalogo, schema e volume
catalog_name = "cinedata_analytics"
schema_name = "tabelas"
volume_name = "inputs"
schema_bronze = "bronze"

# Criação do catalogo e schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

In [0]:
from pyspark.sql.functions import current_timestamp

# Copia os arquivos para o volume
dbutils.fs.cp(
    "file:/Workspace/Users/severojoaoopedro90@gmail.com/projeto-cinedata-analytics/Inputs - Atividade Engenharia de Dados - Bases de Dados/",
    f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/",
    recurse=True
)

# Criação do schema bronze
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_bronze}")

base_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/"

# Mapeando os arquivos para tabelas bronze
mapeamento_tabelas = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

# Apaga as tabelas antes de recriar, evitando duplicação
for nome_tabela in mapeamento_tabelas.values():
    spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.{schema_bronze}.{nome_tabela}")

# Ingestão de verdade
for arquivo, nome_tabela in mapeamento_tabelas.items():
    df = spark.read.csv(
        base_path + arquivo,
        header=True,
        inferSchema=True
    )
    
    df = df.withColumn("ingestion_datetime", current_timestamp())
    
    df.write.mode("append").format("delta").saveAsTable(f"{catalog_name}.{schema_bronze}.{nome_tabela}")
    print(f"Tabela {nome_tabela} criada com sucesso.")

In [0]:
from datetime import datetime, timedelta

# Definindo datas iniciais e finais
data_fim = datetime.today()
data_inicio = data_fim - timedelta(days=7)

# Formatando as datas
data_inicio_formatada = data_inicio.strftime("%m-%d-%Y")
data_fim_formatada = data_fim.strftime("%m-%d-%Y")

# Definindo os parametros
dbutils.widgets.text("data_inicio", data_inicio_formatada)
dbutils.widgets.text("data_fim", data_fim_formatada)

# Recuperando os parametros
data_inicio_param = dbutils.widgets.get("data_inicio")
data_fim_param = dbutils.widgets.get("data_fim")


In [0]:
import requests

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo("
    f"dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_param}'&@dataFinalCotacao='{data_fim_param}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

response = requests.get(url)
dados_json = response.json()

# Verificando se a resposta é válida
if response.status_code == 200:
    print(dados_json)
else:
    print(f"Erro na requisição: {response.status_code}")

In [0]:
 
value = dados_json["value"]

# Criando um dataframe com os dados 
df_cotacao = spark.createDataFrame(value)

# Apagando a tabela caso ela ja exista
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.{schema_bronze}.tb_cotacao_dolar")

# Adicionando uma coluna com a data de ingestão
df_cotacao.write.mode("append").format("delta").saveAsTable(f"{catalog_name}.{schema_bronze}.tb_cotacao_dolar")

print("Tabela tb_cotacao_dolar criada com sucesso.")